In [3]:
import os
import numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
import torch
from transformers import CLIPProcessor, CLIPModel
import timm

In [16]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# Load CLIP model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [5]:
# Load ViT model
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
vit_model.eval()

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity(

In [6]:
# Common utility functions
def mae(a, b):
    return np.mean(np.abs(a - b))

def mse(a, b):
    return np.mean((a - b) ** 2)

def correlation_coefficient(a, b):
    return pearsonr(a, b)[0]

def preprocess_image(image_path, size=(128, 128)):
    img = Image.open(image_path).convert("RGB").resize(size)
    return np.array(img).flatten()

In [7]:
def load_images(image_folder, method="flatten"):
    embeddings = []
    image_paths = []
    for img_file in os.listdir(image_folder):
        if img_file.endswith(('.jpg', '.png', '.jpeg')):
            image_path = os.path.join(image_folder, img_file)
            if method == "flatten":
                embedding = preprocess_image(image_path)
            elif method == "clip":
                embedding = generate_clip_embedding(image_path)
            elif method == "vit":
                embedding = generate_vit_embedding(image_path)
            embeddings.append(embedding)
            image_paths.append(image_path)
    return np.array(embeddings), image_paths

In [8]:
# CLIP embedding generation
def generate_clip_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    embeddings = clip_model.get_image_features(**inputs)
    return embeddings.detach().numpy().flatten()

In [9]:
# ViT embedding generation
def generate_vit_embedding(image_path):
    img = Image.open(image_path).convert("RGB").resize((224, 224))
    img = np.array(img).transpose(2, 0, 1) / 255.0
    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        embeddings = vit_model(img_tensor)
    return embeddings.detach().numpy().flatten()

In [10]:
# Image retrieval
def retrieve_images(query_embedding, embeddings, image_paths, metric="mae", top_k=5):
    results = []
    for i, emb in enumerate(embeddings):
        if metric == "mae":
            score = mae(query_embedding, emb)
        elif metric == "mse":
            score = mse(query_embedding, emb)
        elif metric == "cosine":
            score = cosine_similarity([query_embedding], [emb])[0][0]
        elif metric == "correlation":
            score = correlation_coefficient(query_embedding, emb)
        results.append((image_paths[i], score))
    reverse = metric in ["cosine", "correlation"]
    return sorted(results, key=lambda x: x[1], reverse=reverse)[:top_k]

In [13]:
image_folder = "dataset/images_mr"  
query_image_path = "dataset/images_mr/281.jpg"  

In [17]:
# Process Flatten
print("Using Flatten Embedding:")
flatten_embeddings, flatten_image_paths = load_images(image_folder, method="flatten")
query_flatten_embedding = preprocess_image(query_image_path)
flatten_results = retrieve_images(query_flatten_embedding, flatten_embeddings, flatten_image_paths, metric="mae")
for result in flatten_results:
    print(f"Path: {result[0]}, MAE: {result[1]:.2f}")

# Process CLIP
print("\nUsing CLIP Embedding:")
clip_embeddings, clip_image_paths = load_images(image_folder, method="clip")
query_clip_embedding = generate_clip_embedding(query_image_path)
clip_results = retrieve_images(query_clip_embedding, clip_embeddings, clip_image_paths, metric="cosine")
for result in clip_results:
    print(f"Path: {result[0]}, Cosine Similarity: {result[1]:.2f}")

# Process ViT
print("\nUsing ViT Embedding:")
vit_embeddings, vit_image_paths = load_images(image_folder, method="vit")
query_vit_embedding = generate_vit_embedding(query_image_path)
vit_results = retrieve_images(query_vit_embedding, vit_embeddings, vit_image_paths, metric="cosine")
for result in vit_results:
    print(f"Path: {result[0]}, Cosine Similarity: {result[1]:.2f}")

Using Flatten Embedding:
Path: dataset/images_mr/281.jpg, MAE: 0.00
Path: dataset/images_mr/2399.jpg, MAE: 89.04
Path: dataset/images_mr/5995.jpg, MAE: 89.59
Path: dataset/images_mr/5816.jpg, MAE: 90.33
Path: dataset/images_mr/2287.jpg, MAE: 90.75

Using CLIP Embedding:


KeyboardInterrupt: 